In [ ]:
import json
PARAMS = json.load(open("../params.json", "r"))

SEED = PARAMS["SEED"]
S_MAX = PARAMS["S_MAX"]
S_MIN = PARAMS["S_MIN"]

SAMPLING_SIZE = PARAMS["SAMPLING_SIZE"]
MAXLEN_A = PARAMS["MAXLEN_A"]

HF_TOKEN = PARAMS["HF_TOKEN"] 
MODEL_NAME = PARAMS["MODEL_NAME"] 
USERNAME = PARAMS["USERNAME"] 

ESSAY_SET = PARAMS["ESSAY_SET"]

In [ ]:
CUDA_DEVICE = "cuda:0"
import json
import pandas as pd
import numpy as np
import pickle
from transformers import BertTokenizer, BertModel
import torch
import gc
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from huggingface_hub import login as hf_login
from huggingface_hub import HfApi

In [ ]:
print(torch.cuda.is_available())

torch.cuda.set_device(CUDA_DEVICE)

import gc
torch.cuda.empty_cache()
gc.collect()

In [ ]:
from huggingface_hub import hf_hub_url, cached_download

repo_name = MODEL_NAME
config_file_url = hf_hub_url(f"{USERNAME}/"+repo_name, filename="cls_layer.torch")
value = cached_download(config_file_url)
cls_layer = torch.load(value).cuda()

the_model = BertModel.from_pretrained(f"{USERNAME}/"+repo_name).cuda()
the_tokenizer = BertTokenizer.from_pretrained(f"{USERNAME}/"+repo_name, do_lower_case=False)

class DatasetTaskClassification(Dataset):
    def __init__(self, df, maxlen_A=MAXLEN_A, label=True, tokenizer=None):
        self.df = df
        self.tokenizer = tokenizer
        self.maxlen_A = maxlen_A
        self.label = label
        self.wte = BertModel.from_pretrained(f"{USERNAME}/"+repo_name).cpu().embeddings.word_embeddings

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        score = 0.0
        if self.label:
            score = float(self.df.loc[index, "score"])

        a = self.df.loc[index, "essay"]
        sentence2 = str(a)
        sentence2 = "" if sentence2 == "nan" else sentence2
        sentence2 = sentence2.strip()
                
        tokens2 = self.tokenizer.tokenize(sentence2) if len(sentence2)>0 else ["[UNK]"]

        if len(tokens2) <= self.maxlen_A:
            tokens2 = tokens2 + ['[PAD]' for _ in range(self.maxlen_A - len(tokens2))]
        else:
            tokens2 = tokens2[:self.maxlen_A]
                
        tokens = ["[CLS]"]+tokens2+["[SEP]"]
        tokens_ids = self.tokenizer.convert_tokens_to_ids(tokens)
        tokens_ids_tensor = torch.tensor(tokens_ids)
        attn_mask = (tokens_ids_tensor != 1).long() # [PAD] => 1

        with torch.no_grad():
            # Get input embeddings (e.g., from the model's embedding layer)
            embedding_output = self.wte(tokens_ids_tensor)
        
        return tokens_ids_tensor, embedding_output, attn_mask, score
    
class RegressionModel(nn.Module):
    def __init__(self):
        super(RegressionModel, self).__init__()
        torch.manual_seed(SEED)
        
        self.bert_layer = the_model.cuda()
        self.cls_layer = cls_layer
        self.relu = nn.ReLU(inplace=False)

    def forward(self, input_embeds, attn_masks):

        cont_reps = self.bert_layer(inputs_embeds=input_embeds, attention_mask=attn_masks)
            
        cls_rep = cont_reps.last_hidden_state[:, 0]

        post_relu = self.relu(cls_rep)
        prelogits = self.cls_layer(post_relu)
        
        return prelogits

In [ ]:
df = pd.read_csv(f"../../../data/essay_set_{ESSAY_SET}.csv", index_col=0)
df = df[df["split"] == "test"]
indexs = df.index
df = df.reset_index()

data_set = DatasetTaskClassification(df = df, label=False, tokenizer=the_tokenizer)
data_loader = DataLoader(data_set, batch_size = 1, num_workers = 2, shuffle=False)

In [ ]:
df

In [ ]:
import torch
from captum.attr import IntegratedGradients

In [ ]:
net = RegressionModel()

In [ ]:
import torch
from captum.attr import IntegratedGradients
import re
import matplotlib.pyplot as plt

# Function to split text into clauses based on punctuation
def split_into_clauses(tokens):
    # Ensure 'text' is a string (if it's a list, join into a string)
    if isinstance(tokens, list):
        first = tokens[0]
        text = the_tokenizer.convert_tokens_to_string(tokens[1:-1])
        last = tokens[-1]
    
    # This uses regular expressions to split text into clauses
    # Clauses are defined by sentences or sections separated by punctuation (., ;, !, ?)
    clauses = re.split(r'[.!?;]', text)  # Split by punctuation
    clauses = [first]+[clause.strip() for clause in clauses if clause.strip()]+[last]
    return clauses

def forward_func(input_embeds, attention_mask):
    logits = net(input_embeds, attention_mask)
    return logits

# Define function to compute attributions for each clause
def compute_clause_attributions(input_ids, seq, attn_masks, criterion=max):
    # Split the input sequence into clauses (by tokens)
    clauses = split_into_clauses(the_tokenizer.convert_ids_to_tokens(input_ids[0].cpu().numpy()))

    baseline_seq = torch.full_like(seq, the_tokenizer.pad_token_id).cuda()

    # Initialize Integrated Gradients
    ig = IntegratedGradients(forward_func)

    # Compute attributions (by tokens)
    attributions, _ = ig.attribute(inputs=seq,
                                       baselines=baseline_seq,
                                       additional_forward_args=(attn_masks,),
                                       return_convergence_delta=True)

    torch.cuda.empty_cache()  # Clear the cached memory in CUDA

    # Sum attributions across the embedding dimension to get relevance per token
    attributions = attributions.sum(dim=-1).squeeze(0)  # Sum across embedding dimensions
    attributions = attributions.detach().cpu().numpy()

    # Create a dictionary to hold total attributions by clause
    clause_attributions = {}
    token_idx = 1
    # Loop over each clause and sum the attributions for the tokens within that clause
    for clause in clauses[1:-1]:
        clause_tokens = the_tokenizer.tokenize(clause)
        clause_length = len(clause_tokens)
        clause_attr = criterion(attributions[token_idx:token_idx + clause_length])  # Sum token attributions for the clause
        clause_attributions[clause] = clause_attr
        token_idx += clause_length  # Move to the next clause

    return clause_attributions

In [ ]:
y_true = df["score"].values
pred = pd.read_csv("pred/lf/test.csv", index_col=0)
y_pred = pred["pred"]

m, M = np.min(y_true), np.max(y_true)
scaled_true = (S_MIN + ((pd.Series(y_true) - m) / (M - m)) * (S_MAX - S_MIN)).values

m, M = np.min(y_pred), np.max(y_pred)#np.percentile(y_pred, 1), np.percentile(y_pred, 99) 
scale_preds = (S_MIN + ((pd.Series(y_pred) - m) / (M - m)) * (S_MAX - S_MIN)).values

In [ ]:
df["test"] = scaled_true
df["pred"] = scale_preds

In [ ]:
with open('../../../data/question.json', 'r') as f:
    meta_type = json.load(f)

p = meta_type[str(ESSAY_SET)]["question"]
p

In [ ]:
wte = BertModel.from_pretrained(f"{USERNAME}/"+repo_name).cpu().embeddings.word_embeddings

def preprocess(df, essay_id):
    maxlen_A =MAXLEN_A
    a = df[df["essay_id"] == essay_id].iloc[0]["essay"]
    sentence2 = str(a)
    sentence2 = "" if sentence2 == "nan" else sentence2
    sentence2 = sentence2.strip()
            
    tokens2 = the_tokenizer.tokenize(sentence2) if len(sentence2)>0 else ["[UNK]"]

    if len(tokens2) <= maxlen_A:
        tokens2 = tokens2 + ['[PAD]' for _ in range(maxlen_A - len(tokens2))]
    else:
        tokens2 = tokens2[:maxlen_A]
            
    tokens = ["[CLS]"]+tokens2+["[SEP]"]
    tokens_ids = the_tokenizer.convert_tokens_to_ids(tokens)
    tokens_ids_tensor = torch.tensor([tokens_ids])
    attn_mask = (tokens_ids_tensor != 1).long() # [PAD] => 1

    with torch.no_grad():
        # Get input embeddings (e.g., from the model's embedding layer)
        embedding_output = wte(tokens_ids_tensor)
    
    return tokens_ids_tensor, embedding_output, attn_mask

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'serif' 
matplotlib.rcParams['font.serif'] = ['DejaVu Serif']

def compute_integrated_gradients(essay_id):

    input_ids, seq, attn_masks = preprocess(df, essay_id)

    seq, attn_masks = seq.cuda(), attn_masks.cuda()

    real_score = df[df["essay_id"] == essay_id].iloc[0]["test"]
    predicted_score = df[df["essay_id"] == essay_id].iloc[0]["pred"]

    # Example usage (for a single input sequence)
    seq, attn_masks = seq.cuda(), attn_masks.cuda()  # Your token IDs and attention masks

    # Compute clause attributions
    clause_attributions = compute_clause_attributions(input_ids, seq, attn_masks, max)
    clauses = clause_attributions.items()
    M = max(clause_attributions.values())

    # Define a color map that distinguishes positive and negative attributions
    def attribution_color(attribution, M=1):
        if attribution > 0:
            # Green for positive attributions
            return plt.cm.Greens(attribution/M)
        else:
            # Red for negative attributions
            return plt.cm.Reds(-attribution/M)

    # Combine all clauses into a single essay text for the left side
    essay_text = ". ".join([f"[{i+1}.] {clause[0]}" for i, clause in enumerate(clauses)])+"."

    # Create a figure with two subplots: one for the essay and one for the bar plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 8), dpi=120)

    # Left subplot: essay text with clauses enumerated
    ax1.axis('off')  # Turn off the axis for the left plot
    # Title for the left subplot
    ax1.text(.05+0.18, 1.059-0.28+0.03-0.17+0.025+0.06, 'Essay', ha='center', fontsize=12)#, fontweight='bold')

    # Display the essay text in a text box with wrapping
    ax1.text(.23, 1-0.25+0.03-0.17+0.025+0.06, essay_text, fontsize=10, verticalalignment='top', horizontalalignment='center', 
            wrap=True, bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.0))#, edgecolor="tab:olive"))

    # Display the prompt text
    ax1.text(0.05+0.18, 1.059-0.28+0.03+0.05+0.01, 'Prompt', ha='center', fontsize=12)#, fontweight='bold')
    ax1.text(0.23, 1-0.25+0.03+0.05+0.01, meta_type[str(ESSAY_SET)]["question"], fontsize=10, verticalalignment='top', horizontalalignment='center', 
            wrap=True, bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgrey', alpha=0.5, edgecolor="black"))

    # Add value boxes for real and predicted scores
    ax1.text(0.05-0.3, 1.01-0.055, 'Scores', ha='left', fontsize=12)#, fontweight='bold')
    ax1.text(0.05+0.4-0.165, 1.01-0.07, f'Real:\n{real_score:.1f}', horizontalalignment='left', fontsize=10, 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='tab:gray', alpha=0.3, edgecolor="black"))
    ax1.text(0.05+0.58-0.165, 1.01-0.07, f'Predicted:\n{predicted_score:.1f}', horizontalalignment='left', fontsize=10, 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.4, edgecolor="tab:green"))

    # Add score range annotation
    ax1.text(0.05-0.0, 1.01-0.07, f'Range:\n[{S_MIN}, {S_MAX}]', horizontalalignment='left', fontsize=10, 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.4, edgecolor="tab:blue"))


    # Right subplot: attribution scores with bars
    y_positions = []

    # Right subplot: attribution scores with bars
    for i, (clause, attribution) in enumerate(clauses):
        y_position = len(clauses) - i - 1 * 0.6  # Adjusting the y_position to space out the boxes
        ax2.barh(y_position, attribution, height=0.4, color=attribution_color(attribution, M))
        y_positions.append(y_position)
        # Label the attribution value next to the bar
        ax2.text(attribution, y_position, f'{attribution:.2f}', va='center', fontsize=10, color='black')

    # Set titles and labels for the right subplot
    ax2.set_title('Attribution Scores per Clause', fontsize=12)
    ax2.set_xlabel('Attribution Score', fontsize=10)
    # ax2.set_ylabel('Clause Number', fontsize=10)

    # Customize y-axis labels and ticks
    ax2.tick_params(axis='y', direction='out', length=6, labelright=True, labelleft=False, right=True, left=False)

    # Customize the borders of the bars
    for patch in ax2.patches:
        patch.set_edgecolor(patch._original_facecolor)  # Set the border color to black
        patch.set_linewidth(1.3)  # Set the thickness of the border
        

    # Set custom y-ticks with enumeration 1 to N
    ax2.set_yticks(y_positions)  # Set y-tick positions
    ax2.set_yticklabels([f"[{i+1}.]" for i in range(len(clauses))])  # Set y-tick labels as clause numbers

    # Set the limits and grid for the bar plot
    #ax2.set_xlim(-1, 1)  # Limit x-axis to include both positive and negative scores
    ax2.set_xlim([min(0,min(clause_attributions.values())*1.2), max(clause_attributions.values())*1.2])
    ax2.axvline(x=0, color='black', linestyle='--', alpha=0.5)  # Vertical line for 0 attribution
    ax2.grid(True, axis='x', linestyle='-', alpha=0.7)

    plt.tight_layout(pad=6)
    plt.savefig(f"interpret_{essay_id}.pdf", bbox_inches='tight', transparent=True)
    plt.show()

compute_integrated_gradients(essay_id=21603)

In [ ]:
df["test"]

In [ ]:
# Function to split text into clauses based on punctuation
def split_into_terms(tokens):
    # Ensure 'text' is a string (if it's a list, join into a string)
    terms = [the_tokenizer.convert_tokens_to_string(tokens[i:i+1]).replace("#", "") for i in range(len(tokens))]
    return terms

# Define function to compute attributions for each clause
def compute_term_attributions(input_ids, seq, attn_masks):
    # Split the input sequence into clauses (by tokens)
    terms = split_into_terms(the_tokenizer.convert_ids_to_tokens(input_ids[0].cpu().numpy()))

    baseline_seq = torch.full_like(seq, the_tokenizer.pad_token_id).cuda()

    # Initialize Integrated Gradients
    ig = IntegratedGradients(forward_func)

    # Compute attributions (by tokens)
    attributions, _ = ig.attribute(inputs=seq,
                                       baselines=baseline_seq,
                                       additional_forward_args=(attn_masks,),
                                       return_convergence_delta=True)

    torch.cuda.empty_cache()  # Clear the cached memory in CUDA

    # Sum attributions across the embedding dimension to get relevance per token
    attributions = attributions.sum(dim=-1).squeeze(0)  # Sum across embedding dimensions
    attributions = attributions.detach().cpu().numpy()

    # Create a dictionary to hold total attributions by clause
    term_attributions = {}
    token_idx = 1
    # Loop over each clause and sum the attributions for the tokens within that clause
    for i, term in enumerate(terms[1:-1]):
        term_tokens = the_tokenizer.tokenize(term)
        term_length = len(term_tokens)
        term_attr = sum(attributions[token_idx:token_idx + term_length])  # Sum token attributions for the clause
        term_attributions[i] = (term, term_attr)
        token_idx += term_length  # Move to the next clause

    return term_attributions

In [ ]:
import textwrap

text = """
This is a long piece of text <that> <we> <want> to wrap by words, ensuring that it doesn't split words in the middle of a line.
"""

wrapped_text = textwrap.fill(text, width=40)

print(wrapped_text)


In [ ]:
import matplotlib
from highlight_text import HighlightText, ax_text, fig_text
import textwrap


matplotlib.rcParams['font.family'] = 'serif' 
matplotlib.rcParams['font.serif'] = ['DejaVu Serif']

def compute_integrated_gradients_per_term(essay_id):

    input_ids, seq, attn_masks = preprocess(df, essay_id)

    seq, attn_masks = seq.cuda(), attn_masks.cuda()

    real_score = df[df["essay_id"] == essay_id].iloc[0]["test"]
    predicted_score = df[df["essay_id"] == essay_id].iloc[0]["pred"]

    # Example usage (for a single input sequence)
    seq, attn_masks = seq.cuda(), attn_masks.cuda()  # Your token IDs and attention masks

    # Compute clause attributions
    term_attributions = compute_term_attributions(input_ids, seq, attn_masks)
    term_name = {k: v[0] for k, v in term_attributions.items()}
    term_attrib = {k: v[1] for k, v in term_attributions.items()}

    M = max(term_attrib.values())

    # Define a color map that distinguishes positive and negative attributions
    def attribution_style(attribution, M=1, m=0.01):

        if attribution > m*M:
            # Green for positive attributions
            return {"color": ["black", "white"][attribution/M>0.8], "bbox": {"edgecolor": "lightblue", "facecolor": plt.cm.Blues(attribution/M), "linewidth": 0.5, "pad": 2}} 
        elif attribution < -m*M:
            # Red for negative attributions
            return {"color": ["black", "white"][attribution/M>0.8], "bbox": {"edgecolor": "lightcoral", "facecolor": plt.cm.Reds(-attribution/M), "linewidth": 0.5, "pad": 2}} 
        else:
            return {"color": "black", "bbox": {"edgecolor": "lightgray", "facecolor": "white", "linewidth": 0, "pad": 2}} 

    # Combine all clauses into a single essay text for the left side
    essay_text = " ".join([f"<{term}>" for i, term in term_name.items()])

    # Create a figure with two subplots: one for the essay and one for the bar plot
    fig, ax1 = plt.subplots(1, 1, figsize=(6, 8), dpi=120)

    # Left subplot: essay text with clauses enumerated
    ax1.axis('off')  # Turn off the axis for the left plot
    # Title for the left subplot
    ax1.text(.05+0.18, 1.059-0.28+0.03-0.17+0.025+0.04, 'Essay', ha='center', fontsize=12)#, fontweight='bold')

    # Display the essay text in a text box with wrapping
#     ax1.text(.23, 1-0.25+0.03-0.17+0.025+0.06, essay_text, fontsize=10, verticalalignment='top', horizontalalignment='center', 
#             wrap=True, bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.0))#, edgecolor="tab:olive"))


    HighlightText(x=.23, y=1-0.25+0.03-0.17+0.025+0.04,
              s=textwrap.fill(essay_text, width=100), #'The weather is <sunny>\nYesterday it was <cloudy>', #essay_text
              va='top', ha='center',
              highlight_textprops=[attribution_style(attribution, M) for i, attribution in term_attrib.items()],
              textalign='center',
              ax=ax1)
    
    # Display the prompt text
    ax1.text(0.05+0.18, 1.059-0.28+0.03+0.05+0.01, 'Prompt', ha='center', fontsize=12)#, fontweight='bold')
    ax1.text(0.23, 1-0.25+0.03+0.05+0.01, meta_type[str(ESSAY_SET)]["question"], fontsize=10, verticalalignment='top', horizontalalignment='center', 
            wrap=True, bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgrey', alpha=0.5, edgecolor="black"))

    # Add value boxes for real and predicted scores
    ax1.text(0.05-0.3, 1.01-0.055, 'Scores', ha='left', fontsize=12)#, fontweight='bold')
    ax1.text(0.05+0.4-0.165, 1.01-0.07, f'Real:\n{real_score:.1f}', horizontalalignment='left', fontsize=10, 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='tab:gray', alpha=0.3, edgecolor="black"))
    ax1.text(0.05+0.58-0.165, 1.01-0.07, f'Predicted:\n{predicted_score:.1f}', horizontalalignment='left', fontsize=10, 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.4, edgecolor="tab:green"))

    # Add score range annotation
    ax1.text(0.05-0.0, 1.01-0.07, f'Range:\n[{S_MIN}, {S_MAX}]', horizontalalignment='left', fontsize=10, 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.4, edgecolor="tab:blue"))

    plt.tight_layout(pad=6)
    plt.savefig(f"interpret_{essay_id}.pdf", bbox_inches='tight', transparent=True)
    plt.show()

compute_integrated_gradients_per_term(essay_id=21603)

In [ ]:
df.loc[(df["test"] - df["pred"]).abs().sort_values(ascending=True).index]